In [ ]:
import json
import hashlib
import shutil
from pyprojroot import here
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from scaper_ai import settings, create_vector_store, create_retriever

In [ ]:
shutil.rmtree(here("db/example_chroma_db"))

# Example retrieval pipeline
This notebook demonstrates how to build and query a Chroma vector store using the shared `scaper_ai` factory functions. The example uses `example-collection` and excludes scraping.

In [ ]:
DATA_PATH = "data/rip/20260615"

In [ ]:
data_path = here(DATA_PATH)

documents = []
for file in data_path.glob("*.txt"):
        try:
            with open(file, "r", encoding="utf-8") as f:
                content = f.read()

            doc = Document(page_content=content, metadata={"source": str(file)})
            documents.append(doc)

        except Exception as e:
            ...

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500, add_start_index=True)
split_docs = splitter.split_documents(documents)

len(split_docs)

In [ ]:
vector_store = create_vector_store(
    collection_name="example-collection",
    persist_directory=here("db/example_chroma_db"),
)

To avoid document duplication, we create a unique ID based off of the documents content.

In [ ]:
def hash_doc(document: Document) -> str:
    doc = str(document.model_dump())

    return hashlib.sha256(doc.encode()).hexdigest()

ids = vector_store.add_documents(
    documents=split_docs,
    ids=[hash_doc(doc) for doc in split_docs]
)

In [ ]:
retriever = create_retriever(vector_store=vector_store)

In [ ]:
query = "What are the ideal water parameters for a tropical community tank?"
results = retriever.invoke(query)

results

This pipeline loads existing text files, splits them into chunks, creates a Chroma store in `db/example_chroma_db`, and queries the `example-collection` retriever.